In [256]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from sklearn.preprocessing import MultiLabelBinarizer, LabelEncoder
from sklearn.metrics import classification_report, accuracy_score
import pickle

In [258]:
# Load cleaned dataset
df = pd.read_csv("data/cleaned_disease_symptoms.csv")

In [260]:
# Extract symptom columns
symptom_cols = [col for col in df.columns if col.lower().startswith("symptom")]

In [262]:
# Combine symptoms into list
df["Symptom_List"] = df[symptom_cols].values.tolist()
df["Symptom_List"] = df["Symptom_List"].apply(
    lambda x: list(set([s.strip().lower() for s in x if isinstance(s, str) and s.strip()]))
)

In [264]:
# Drop rows without symptoms
df = df[df["Symptom_List"].map(len) > 0]

In [266]:
# Binarize symptoms
mlb = MultiLabelBinarizer()
X = mlb.fit_transform(df["Symptom_List"])

In [268]:
# Encode target (disease)
le = LabelEncoder()
y = le.fit_transform(df["Disease"])

In [270]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [272]:
# Cross-validation strategy
cv_strategy = StratifiedKFold(n_splits=2, shuffle=True, random_state=42)

In [274]:
# Initialize models
rf = RandomForestClassifier(random_state=42)
xgb = XGBClassifier(eval_metric='mlogloss', random_state=42)
svm = SVC(probability=True, random_state=42)
nb = GaussianNB()

In [276]:
# Hyperparameter tuning
rf_grid = GridSearchCV(rf, {'n_estimators': [100, 150], 'max_depth': [None, 10]}, cv=cv_strategy, n_jobs=-1)
xgb_grid = GridSearchCV(xgb, {'n_estimators': [100, 150], 'max_depth': [3, 5]}, cv=cv_strategy, n_jobs=-1)
svm_grid = GridSearchCV(svm, {'C': [1, 10], 'kernel': ['linear', 'rbf']}, cv=cv_strategy, n_jobs=-1)
nb_grid = GridSearchCV(nb, {'var_smoothing': np.logspace(0, -9, 10)}, cv=cv_strategy, n_jobs=-1)

In [278]:
# Train models
print("🔍 Tuning Random Forest...")
rf_grid.fit(X_train, y_train)

🔍 Tuning Random Forest...


GridSearchCV(cv=StratifiedKFold(n_splits=2, random_state=42, shuffle=True),
             estimator=RandomForestClassifier(random_state=42), n_jobs=-1,
             param_grid={'max_depth': [None, 10], 'n_estimators': [100, 150]})

In [280]:
print("🔍 Tuning XGBoost...")
xgb_grid.fit(X_train, y_train)

🔍 Tuning XGBoost...


GridSearchCV(cv=StratifiedKFold(n_splits=2, random_state=42, shuffle=True),
             estimator=XGBClassifier(base_score=None, booster=None,
                                     callbacks=None, colsample_bylevel=None,
                                     colsample_bynode=None,
                                     colsample_bytree=None, device=None,
                                     early_stopping_rounds=None,
                                     enable_categorical=False,
                                     eval_metric='mlogloss', feature_types=None,
                                     feature_weights=None, gamma=None,
                                     grow...
                                     interaction_constraints=None,
                                     learning_rate=None, max_bin=None,
                                     max_cat_threshold=None,
                                     max_cat_to_onehot=None,
                                     max_delta_step=None, max_depth=None,
                                     max_leaves=None, min_child_weight=None,
                                     missing=nan, monotone_constraints=None,
                                     multi_strategy=None, n_estimators=None,
                                     n_jobs=None, num_parallel_tree=None, ...),
             n_jobs=-1,
             param_grid={'max_depth': [3, 5], 'n_estimators': [100, 150]})

In [282]:
print("🔍 Tuning SVM...")
svm_grid.fit(X_train, y_train)

🔍 Tuning SVM...


GridSearchCV(cv=StratifiedKFold(n_splits=2, random_state=42, shuffle=True),
             estimator=SVC(probability=True, random_state=42), n_jobs=-1,
             param_grid={'C': [1, 10], 'kernel': ['linear', 'rbf']})

In [284]:
print("🔍 Tuning Naive Bayes...")
nb_grid.fit(X_train, y_train)

🔍 Tuning Naive Bayes...


GridSearchCV(cv=StratifiedKFold(n_splits=2, random_state=42, shuffle=True),
             estimator=GaussianNB(), n_jobs=-1,
             param_grid={'var_smoothing': array([1.e+00, 1.e-01, 1.e-02, 1.e-03, 1.e-04, 1.e-05, 1.e-06, 1.e-07,
       1.e-08, 1.e-09])})

In [286]:
# Best models
rf_best = rf_grid.best_estimator_
xgb_best = xgb_grid.best_estimator_
svm_best = svm_grid.best_estimator_
nb_best = nb_grid.best_estimator_

In [288]:
# Ensemble model
ensemble = VotingClassifier(estimators=[
    ('rf', rf_best),
    ('xgb', xgb_best),
    ('svm', svm_best),
    ('nb', nb_best)
], voting='soft')

In [290]:
# Fit ensemble
print("\n🤖 Training Ensemble Model...")
ensemble.fit(X_train, y_train)


🤖 Training Ensemble Model...


VotingClassifier(estimators=[('rf', RandomForestClassifier(random_state=42)),
                             ('xgb',
                              XGBClassifier(base_score=None, booster=None,
                                            callbacks=None,
                                            colsample_bylevel=None,
                                            colsample_bynode=None,
                                            colsample_bytree=None, device=None,
                                            early_stopping_rounds=None,
                                            enable_categorical=False,
                                            eval_metric='mlogloss',
                                            feature_types=None,
                                            feature_weights=None, gamma=None,
                                            grow...
                                            max_cat_threshold=None,
                                            max_cat_to_onehot=None,
                                            max_delta_step=None, max_depth=3,
                                            max_leaves=None,
                                            min_child_weight=None, missing=nan,
                                            monotone_constraints=None,
                                            multi_strategy=None,
                                            n_estimators=100, n_jobs=None,
                                            num_parallel_tree=None, ...)),
                             ('svm',
                              SVC(C=1, kernel='linear', probability=True,
                                  random_state=42)),
                             ('nb', GaussianNB(var_smoothing=1.0))],
                 voting='soft')

In [292]:
# Evaluate ensemble
y_pred = ensemble.predict(X_test)
print("\n🎯 Ensemble Accuracy:", accuracy_score(y_test, y_pred))
print("\n📊 Classification Report (Ensemble):\n", classification_report(y_test, y_pred, target_names=le.classes_))


🎯 Ensemble Accuracy: 1.0

📊 Classification Report (Ensemble):
                                          precision    recall  f1-score   support

(vertigo) Paroymsal  Positional Vertigo       1.00      1.00      1.00         1
                                   AIDS       1.00      1.00      1.00         1
                                   Acne       1.00      1.00      1.00         1
                    Alcoholic hepatitis       1.00      1.00      1.00         2
                                Allergy       1.00      1.00      1.00         1
                              Arthritis       1.00      1.00      1.00         1
                       Bronchial Asthma       1.00      1.00      1.00         1
                   Cervical spondylosis       1.00      1.00      1.00         1
                            Chicken pox       1.00      1.00      1.00         2
                    Chronic cholestasis       1.00      1.00      1.00         2
                            Common Cold     

In [294]:
# Compare individual model accuracies
models = {
    "Random Forest": rf_best,
    "XGBoost": xgb_best,
    "SVM": svm_best,
    "Naive Bayes": nb_best,
    "Ensemble": ensemble
}

In [296]:
print("\n📈 Individual Model Accuracies:")
for name, model in models.items():
    pred = model.predict(X_test)
    acc = accuracy_score(y_test, pred)
    print(f"{name}: {acc:.4f}")


📈 Individual Model Accuracies:
Random Forest: 1.0000
XGBoost: 0.8852
SVM: 1.0000
Naive Bayes: 1.0000
Ensemble: 1.0000


In [298]:
# XGBoost classification report
print("\n📊 Classification Report (XGBoost):\n")
y_pred_xgb = xgb_best.predict(X_test)
print(classification_report(y_test, y_pred_xgb, target_names=le.classes_))


📊 Classification Report (XGBoost):

                                         precision    recall  f1-score   support

(vertigo) Paroymsal  Positional Vertigo       1.00      1.00      1.00         1
                                   AIDS       1.00      1.00      1.00         1
                                   Acne       1.00      1.00      1.00         1
                    Alcoholic hepatitis       1.00      0.50      0.67         2
                                Allergy       1.00      1.00      1.00         1
                              Arthritis       0.00      0.00      0.00         1
                       Bronchial Asthma       1.00      1.00      1.00         1
                   Cervical spondylosis       1.00      1.00      1.00         1
                            Chicken pox       1.00      1.00      1.00         2
                    Chronic cholestasis       0.50      1.00      0.67         2
                            Common Cold       1.00      1.00      1.00 

C:\Users\ayush\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\ayush\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\ayush\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [300]:
# Save models
with open("model/ensemble_model.pkl", "wb") as f:
    pickle.dump({'model': ensemble, 'label_encoder': le, 'mlb': mlb}, f)

with open("model/xgboost_model.pkl", "wb") as f:
    pickle.dump({'model': xgb_best, 'label_encoder': le, 'mlb': mlb}, f)

In [302]:
print("\n💾 Models saved as 'ensemble_model.pkl' and 'xgboost_model.pkl'")


💾 Models saved as 'ensemble_model.pkl' and 'xgboost_model.pkl'
